# GOT-OCR-2.0 inference on `image_text_v4.2/columns` (Google Colab)

Runs `stepfun-ai/GOT-OCR-2.0-hf` (pretrained, no fine-tuning) over every **column image** produced by
the `image_text_v4.2` preprocessing pipeline (`image_text_v4.2/columns/<page>/<COLUMN>_clean.png`).

Each column strip already spans the full data area of one printed column (grid lines removed), so it
is fed to GOT **as a whole image** (no per-row splitting -- that requires the original page's detected
row lines, which are not needed here since we work directly off the saved column crops).

**Runs on Colab.** Mount your Drive, and make sure one of these is true before running:
- `image_text_v4.2/` already exists under `/content/drive/MyDrive/MeenakshiPublic/` (or directly under
  `/content/drive/MyDrive/`), **or**
- `column_output.zip` (the zipped `image_text_v4.2` folder) is sitting in one of those same Drive
  locations, or in `/content/` -- it will be unzipped automatically, **or**
- neither is found: the notebook will prompt you to upload `column_output.zip` directly.

Use a GPU runtime (`Runtime -> Change runtime type -> GPU`) -- GOT-OCR-2.0 on CPU is far too slow for
this many images.

Outputs (written back to the same `image_text_v4.2` folder, on Drive if that's where it came from):
- `columns_ocr_whole_GOT.json` -- full nested result (page -> column -> text/confidence)
- `columns_ocr_whole_GOT.csv` -- flat table, one row per (page, column)
- `structured_whole_GOT/<page>.csv` -- one row per page, columns laid out side by side

In [ ]:
# Colab already ships torch/opencv/pandas/matplotlib; pin transformers to the range that supports
# GOT-OCR-2.0 (added in 4.49) and stays below 5.0 (a tokenizer-backend regression there wrongly demands
# sentencepiece/tiktoken even when they're installed). accelerate is needed for device_map="auto".
#
# NOTE: if this is the FIRST time sentencepiece/tiktoken get installed in this runtime, RESTART THE
# RUNTIME and re-run from the top -- transformers caches "not available" for the tokenizer backend at
# import time, so a same-session install is invisible to it until the process restarts.
!pip install -q -U "transformers>=4.49,<5.0" accelerate sentencepiece tiktoken opencv-python-headless

In [ ]:
import os
import re
import sys
import json
import time
import math
import zipfile
from datetime import datetime

import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image

import torch
from transformers import AutoProcessor, AutoModelForImageTextToText

IN_COLAB = "google.colab" in sys.modules

print("Running in Colab:", IN_COLAB)
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("WARNING: no GPU detected -- go to Runtime -> Change runtime type -> GPU. "
          "GOT-OCR-2.0 on CPU is far too slow for this many images.")

In [ ]:
DRIVE_ROOT = "/content/drive/MyDrive/MeenakshiPublic"   # same Drive folder the v4.2 pipeline notebook writes to

def mount_drive():
    """Mount Google Drive once. Safe to call again -- a no-op once already mounted."""
    if not IN_COLAB:
        return
    from google.colab import drive
    if not os.path.isdir("/content/drive/MyDrive"):
        drive.mount("/content/drive")

mount_drive()

In [ ]:
# ---------------------------------------------------------------------------
# Config -- BASE_DIR is this Drive project's OCR output folder, the same one the
# v4.2 pipeline notebook writes to (DRIVE_ROOT + "/ocr_output_v4_column").
# ---------------------------------------------------------------------------
DEFAULT_BASE_DIR = "/content/drive/MyDrive/MeenakshiPublic/ocr_output_v4_column"


def find_image_text_dir(folder_name="image_text_v4.2"):
    """BASE_DIR resolution, in order:
      1. DEFAULT_BASE_DIR on Drive (already mounted above) -- the normal case
      2. a folder named `folder_name` under /content, DRIVE_ROOT, or straight under My Drive
      3. a column_output.zip in any of those same places -> unzipped once into /content
      4. (Colab only) prompts to upload column_output.zip directly
    """
    if os.path.isdir(DEFAULT_BASE_DIR):
        return DEFAULT_BASE_DIR

    search_roots = ["/content", DRIVE_ROOT, "/content/drive/MyDrive"] if IN_COLAB else [os.getcwd()]
    for root in search_roots:
        candidate = os.path.join(root, folder_name)
        if os.path.isdir(candidate):
            return candidate

    if not IN_COLAB:
        raise FileNotFoundError(
            f"Could not locate {DEFAULT_BASE_DIR} or a '{folder_name}' folder under {os.getcwd()}. "
            "Set BASE_DIR manually to its absolute path."
        )

    zip_candidates = [
        f"{DRIVE_ROOT}/column_output.zip",
        "/content/drive/MyDrive/column_output.zip",
        "/content/column_output.zip",
    ]
    for zpath in zip_candidates:
        if os.path.isfile(zpath):
            print(f"Extracting {zpath} -> /content/{folder_name} ...")
            with zipfile.ZipFile(zpath) as zf:
                zf.extractall("/content")
            candidate = os.path.join("/content", folder_name)
            if os.path.isdir(candidate):
                return candidate

    print(f"Could not find {DEFAULT_BASE_DIR}, a '{folder_name}' folder, or a column_output.zip on Drive/Colab.")
    print("Upload column_output.zip now (the zipped OCR output folder) ...")
    from google.colab import files
    uploaded = files.upload()
    for fname in uploaded:
        if fname.lower().endswith(".zip"):
            with zipfile.ZipFile(fname) as zf:
                zf.extractall("/content")
            candidate = os.path.join("/content", folder_name)
            if os.path.isdir(candidate):
                return candidate

    raise FileNotFoundError(
        f"Still could not locate the OCR output folder. Set BASE_DIR manually to its path "
        f"(e.g. '{DEFAULT_BASE_DIR}')."
    )


BASE_DIR       = find_image_text_dir("image_text_v4.2")
COLUMNS_DIR    = os.path.join(BASE_DIR, "columns")
OUTPUT_DIR     = BASE_DIR
STRUCTURED_DIR = os.path.join(BASE_DIR, "structured_whole_GOT")

MODEL_ID = "stepfun-ai/GOT-OCR-2.0-hf"

MAX_NEW_TOKENS   = 512   # ceiling per column strip; generation stops earlier via the <|im_end|> stop string
NO_REPEAT_NGRAM  = 6     # a verbatim 6-token repeat is always a degenerate loop, never real handwriting
MIN_INK_PIXELS   = 20    # a strip with fewer ink pixels than this is treated as blank

os.makedirs(STRUCTURED_DIR, exist_ok=True)
print("Base dir   :", BASE_DIR)
print("Columns dir:", COLUMNS_DIR)
print("Output dir :", OUTPUT_DIR)


In [ ]:
# ---------------------------------------------------------------------------
# Discover every column image: one entry per (page, column), preferring the
# grid-removed "_clean.png" crop and falling back to the raw crop if missing.
# ---------------------------------------------------------------------------
def discover_columns(columns_dir):
    items = []
    for page in sorted(os.listdir(columns_dir)):
        page_dir = os.path.join(columns_dir, page)
        if not os.path.isdir(page_dir):
            continue
        files = os.listdir(page_dir)
        clean = sorted(f for f in files if f.endswith("_clean.png"))
        seen = set()
        for f in clean:
            col_name = f[: -len("_clean.png")]
            items.append({"page": page, "column": col_name, "path": os.path.join(page_dir, f)})
            seen.add(col_name)
        # fall back to the raw crop for any column that has no _clean variant
        for f in sorted(files):
            if not f.endswith(".png") or f.endswith("_clean.png"):
                continue
            col_name = f[: -len(".png")]
            if col_name not in seen:
                items.append({"page": page, "column": col_name, "path": os.path.join(page_dir, f)})
    return items

COLUMN_IMAGES = discover_columns(COLUMNS_DIR)
print(f"Found {len(COLUMN_IMAGES)} column images across "
      f"{len(sorted(set(c['page'] for c in COLUMN_IMAGES)))} pages")
for page in sorted(set(c["page"] for c in COLUMN_IMAGES)):
    cols = [c["column"] for c in COLUMN_IMAGES if c["page"] == page]
    print(f"  {page}: {len(cols)} columns -> {cols}")

In [ ]:
# Preview the first column image
if COLUMN_IMAGES:
    preview = COLUMN_IMAGES[0]
    img = Image.open(preview["path"])
    print(f"{preview['page']} / {preview['column']}  ({img.size[0]}x{img.size[1]})")
    plt.figure(figsize=(4, 10))
    plt.imshow(img, cmap="gray")
    plt.axis("off")
    plt.title(f"{preview['page']} / {preview['column']}")
    plt.show()
else:
    print("No column images found -- check COLUMNS_DIR.")

## Load GOT-OCR-2.0

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Loading GOT-OCR-2.0 ...")

processor = AutoProcessor.from_pretrained(MODEL_ID)
model = AutoModelForImageTextToText.from_pretrained(
    MODEL_ID,
    device_map="auto",
    dtype=torch.float16 if device == "cuda" else torch.float32,
)
model.eval()
print(f"Model loaded on {device}")

In [ ]:
# ---------------------------------------------------------------------------
# Helpers: trim a column strip down to its ink (drops excess white margin),
# detect the degenerate repetition loop GOT can fall into on hard images, and
# run one image through the model.
# ---------------------------------------------------------------------------
def load_gray(path):
    img = cv2.imread(path, cv2.IMREAD_GRAYSCALE)
    if img is None:
        raise FileNotFoundError(path)
    return img


def ink_pixel_count(gray):
    _, mask = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)
    return int(np.count_nonzero(mask))


def trim_to_ink(gray, margin_ratio=0.03, min_ink_pixels=MIN_INK_PIXELS):
    """Crop to the bounding box of the ink plus a small white margin. None if (almost) no ink."""
    _, mask = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)
    mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, np.ones((2, 2), np.uint8))
    if np.count_nonzero(mask) < min_ink_pixels:
        return None
    ys, xs = np.nonzero(mask)
    trimmed = gray[ys.min(): ys.max() + 1, xs.min(): xs.max() + 1]
    m = int(min(trimmed.shape) * margin_ratio) + 6
    return cv2.copyMakeBorder(trimmed, m, m, m, m, cv2.BORDER_CONSTANT, value=255)


def is_degenerate(text):
    """Repetition loop GOT can fall into on an unreadable image (confidence alone won't catch it)."""
    toks = text.split()
    if len(toks) >= 8 and len(set(toks[-8:])) <= 2:
        return True
    squashed = re.sub(r"\s+", "", text)
    if len(squashed) >= 12 and max((squashed.count(ch) for ch in set(squashed)), default=0) / len(squashed) > 0.6:
        return True
    return False


def got_ocr(gray, max_new_tokens=MAX_NEW_TOKENS, no_repeat_ngram=NO_REPEAT_NGRAM):
    """Run GOT-OCR-2.0 on one grayscale numpy image -> (text, confidence).
    confidence is the mean per-token generation probability, a simple proxy for model certainty."""
    image = Image.fromarray(gray).convert("RGB")
    inputs = processor(image, return_tensors="pt")
    inputs = {k: v.to(model.device) if hasattr(v, "to") else v for k, v in inputs.items()}

    with torch.inference_mode():
        out = model.generate(
            **inputs,
            do_sample=False,
            tokenizer=processor.tokenizer,
            stop_strings="<|im_end|>",
            max_new_tokens=max_new_tokens,
            no_repeat_ngram_size=no_repeat_ngram,
            output_scores=True,
            return_dict_in_generate=True,
        )

    new_ids = out.sequences[0][inputs["input_ids"].shape[1]:]
    text = processor.decode(new_ids, skip_special_tokens=True).strip()

    special = set(processor.tokenizer.all_special_ids)
    probs = [torch.softmax(step[0], dim=-1)[t].item() for step, t in zip(out.scores, new_ids) if t.item() not in special]
    confidence = float(np.mean(probs)) if probs else None

    if is_degenerate(text):
        return "", None, text[:80]
    return text, (round(confidence, 4) if confidence is not None else None), None

## Try it on one column image

In [ ]:
if COLUMN_IMAGES:
    sample = COLUMN_IMAGES[0]
    gray = load_gray(sample["path"])
    trimmed = trim_to_ink(gray)
    t0 = time.time()
    text, confidence, raw = got_ocr(trimmed if trimmed is not None else gray)
    print(f"{sample['page']} / {sample['column']}  ({time.time() - t0:.1f}s)")
    print("Confidence:", confidence)
    print("OCR text:")
    print(text)
    if raw:
        print("(suppressed degenerate output:", raw, ")")

## Run GOT-OCR-2.0 on every column image

In [ ]:
results = {}   # page -> {column: {crop_file, text, confidence, ink_pixels, seconds}}
t_start = time.time()

for i, item in enumerate(COLUMN_IMAGES, start=1):
    page, col, path = item["page"], item["column"], item["path"]
    print(f"[{i}/{len(COLUMN_IMAGES)}] {page} / {col} ...", end=" ", flush=True)
    t0 = time.time()
    try:
        gray = load_gray(path)
        ink = ink_pixel_count(gray)
        trimmed = trim_to_ink(gray)
        if trimmed is None:
            text, confidence = "", None
        else:
            text, confidence, _raw = got_ocr(trimmed)
        elapsed = time.time() - t0
        results.setdefault(page, {})[col] = {
            "crop_file": os.path.relpath(path, BASE_DIR).replace(os.sep, "/"),
            "text": text,
            "confidence": confidence,
            "ink_pixels": ink,
            "seconds": round(elapsed, 2),
        }
        print(f"done ({elapsed:.1f}s, conf={confidence})")
    except Exception as e:
        elapsed = time.time() - t0
        results.setdefault(page, {})[col] = {
            "crop_file": os.path.relpath(path, BASE_DIR).replace(os.sep, "/"),
            "text": "",
            "confidence": None,
            "ink_pixels": None,
            "seconds": round(elapsed, 2),
            "error": f"{type(e).__name__}: {e}",
        }
        print(f"ERROR: {e}")

total_elapsed = time.time() - t_start
print(f"\nDone. Processed {len(COLUMN_IMAGES)} column images in {total_elapsed / 60:.1f} min.")

In [ ]:
# ---------------------------------------------------------------------------
# Save results: nested JSON + flat CSV
# ---------------------------------------------------------------------------
json_path = os.path.join(OUTPUT_DIR, "columns_ocr_whole_GOT.json")
csv_path = os.path.join(OUTPUT_DIR, "columns_ocr_whole_GOT.csv")

payload = {
    "meta": {
        "engine": "got",
        "model": MODEL_ID,
        "split": "whole",
        "crop_kind": "clean",
        "max_new_tokens": MAX_NEW_TOKENS,
        "seconds": round(total_elapsed, 1),
        "created": datetime.now().isoformat(timespec="seconds"),
    },
    "pages": results,
}
with open(json_path, "w", encoding="utf-8") as f:
    json.dump(payload, f, ensure_ascii=False, indent=2)

flat_rows = [
    {"page": page, "column": col, **info}
    for page, cols in results.items()
    for col, info in cols.items()
]
results_df = pd.DataFrame(flat_rows)
results_df.to_csv(csv_path, index=False, encoding="utf-8-sig")

print("Saved JSON:", json_path)
print("Saved CSV :", csv_path)
results_df

## Per-image CSV: image name -> OCR text + confidence

One row per column image (the actual file OCR was run on), independent of the page/column
grouping above -- just the image file name, what GOT read from it, and the confidence.

In [ ]:
# ---------------------------------------------------------------------------
# Per-image CSV: image_name, ocr_text, confidence -- one row per column image
# ---------------------------------------------------------------------------
image_ocr_rows = []
for item in COLUMN_IMAGES:
    page, col, path = item["page"], item["column"], item["path"]
    info = results.get(page, {}).get(col, {})
    image_ocr_rows.append({
        "image_name": os.path.basename(path),
        "page": page,
        "column": col,
        "ocr_text": info.get("text", ""),
        "confidence": info.get("confidence"),
    })

image_ocr_df = pd.DataFrame(image_ocr_rows)
image_ocr_csv_path = os.path.join(OUTPUT_DIR, "image_ocr_results.csv")
image_ocr_df.to_csv(image_ocr_csv_path, index=False, encoding="utf-8-sig")

print(f"Saved {len(image_ocr_df)} rows -> {image_ocr_csv_path}")
image_ocr_df


## Structured per-page tables

One row per page, its columns laid out side by side (whole-column mode reads each column as a single
block of text, so there is exactly one "row" per page here).

In [ ]:
structured_tables = {}
for page, cols in results.items():
    row = {"page": page}
    conf_row = {}
    for col, info in cols.items():
        row[col] = info.get("text", "")
        conf_row[f"{col} (conf)"] = info.get("confidence")
    df = pd.DataFrame([{**row, **conf_row}])
    structured_tables[page] = df
    out_path = os.path.join(STRUCTURED_DIR, f"{re.sub(r'[^A-Za-z0-9_.-]+', '_', page)}.csv")
    df.to_csv(out_path, index=False, encoding="utf-8-sig")
    print(f"{page}: saved {out_path}")

print("\nSaved", len(structured_tables), "structured page tables to", STRUCTURED_DIR)

In [ ]:
# ---------------------------------------------------------------------------
# Summary
# ---------------------------------------------------------------------------
has_text = results_df["text"].astype(str).str.strip() != "" if len(results_df) else pd.Series(dtype=bool)
mean_conf = results_df.loc[has_text, "confidence"].mean() if has_text.any() else float("nan")

print(f"Pages processed   : {len(results)}")
print(f"Columns processed : {len(results_df)}")
print(f"Columns with text : {int(has_text.sum())}")
print(f"Mean confidence   : {mean_conf:.3f}" if not math.isnan(mean_conf) else "Mean confidence   : n/a")
print(f"Total time        : {total_elapsed / 60:.1f} min")

for page, df in structured_tables.items():
    print(f"\n=== {page} ===")
    display(df[[c for c in df.columns if not c.endswith("(conf)")]])